# 02 · A CNN from Scratch with Keras
Keras is installed in Colab · Runtime ➜ Change runtime type ➜ GPU (optional, makes training faster)

1. From the convolution-basics notebook to Keras: one `Conv2D` layer = our NumPy convolution
2. Our own `Conv2D` and `MaxPool2D` as Keras layers, and Keras learning a filter
3. Build and train a CNN from scratch (random start) on MNIST
4. CNN vs MLP: shifted digits and scrambled pixels
5. Look inside: learned filters and feature maps
6. Experiments: filters, kernel size, depth, pooling — and little data
7. Use case (a): sorting product photos for an online shop (Fashion-MNIST)
8. Use case (b): reading a handwritten number

In [ ]:
import time
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt

tf.get_logger().setLevel("ERROR")
keras.utils.set_random_seed(42)
np.set_printoptions(precision=2, suppress=True, linewidth=120)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = (x_train / 255.0).astype("float32")[..., None]           # (N, 28, 28, 1): height, width, channels
x_test = (x_test / 255.0).astype("float32")[..., None]
print("train", x_train.shape, "test", x_test.shape)

---
# Part 1: from NumPy to Keras — one `Conv2D` layer
In the convolution-basics notebook we wrote the convolution in NumPy. A Keras `Conv2D` layer does **exactly** the same thing: give it the Sobel vertical-edge kernel as its weights and compare.

In [ ]:
def conv2d(image, kernel, padding=0, stride=1):
    # the NumPy convolution from the convolution-basics notebook
    if padding:
        image = np.pad(image, padding)
    k = kernel.shape[0]
    windows = np.lib.stride_tricks.sliding_window_view(image, (k, k))[::stride, ::stride]
    return np.einsum("ijkl,kl->ij", windows, kernel)

sobel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype="float32")
digit = x_test[0, :, :, 0]

edge_layer = layers.Conv2D(1, 3, padding="same")                     # 1 filter, 3×3, padding 1
edge_layer.build((None, 28, 28, 1))
edge_layer.set_weights([sobel[:, :, None, None], np.zeros(1)])       # weights (3, 3, in 1, out 1), bias (1,)
keras_map = edge_layer(x_test[:1]).numpy()[0, :, :, 0]
numpy_map = conv2d(digit, sobel, padding=1)
print("Keras Conv2D output == our NumPy convolution?", np.allclose(keras_map, numpy_map, atol=1e-5))

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, img, title in [(axes[0], digit, "digit"), (axes[1], numpy_map, "NumPy conv2d"), (axes[2], keras_map, "Keras Conv2D")]:
    ax.imshow(img, cmap="gray_r" if title == "digit" else "RdBu_r")
    ax.set(title=title, xticks=[], yticks=[])
plt.show()

The Keras arguments are the same knobs: `filters` (how many feature maps), `kernel_size` (k), `strides` (s), `padding` (`"valid"` = none, `"same"` = keep the size). Check the output sizes and weight counts against the formulas $\frac{n + 2p - k}{s} + 1$ and $k \cdot k \cdot C_{in} \cdot C_{out} + C_{out}$:

In [ ]:
print(f"{'layer':>40} | {'output':>14} | {'weights':>7} | {'formula':>17}")
print("-" * 88)
for c_in, f, k, s, pad in [(1, 8, 3, 1, "valid"), (1, 8, 3, 1, "same"), (1, 8, 3, 2, "same"), (8, 16, 3, 1, "same"), (16, 32, 5, 1, "valid")]:
    layer = layers.Conv2D(f, k, strides=s, padding=pad)
    out = layer(np.zeros((1, 28, 28, c_in), dtype="float32"))
    print(f"{f'Conv2D({f}, {k}, strides={s}, padding={pad!r})':>40} | {str(tuple(out.shape[1:])):>14} | "
          f"{layer.count_params():>7,} | {f'{k}·{k}·{c_in}·{f} + {f} = {k * k * c_in * f + f:,}':>17}")

---
# Part 2: our own `Conv2D` as a Keras layer
The trick that makes convolution fast (**im2col**): cut out **every** k×k×C window, flatten each into a row, then do **one matrix multiply** with the filters reshaped to (k·k·C, number of filters).
```
image (N, H, W, C) ─► windows (N, H', W', k·k·C) ─► @ filters (k·k·C, F) ─► + bias ─► (N, H', W', F)
```
We write only the forward pass (`call`): Keras records it and computes the backward pass for us — the same gradients we derived by hand in the convolution-basics notebook.

In [ ]:
class OurConv2D(layers.Layer):
    def __init__(self, filters, kernel_size=3, stride=1, padding="same", **kwargs):
        super().__init__(**kwargs)
        self.filters, self.k, self.stride, self.padding = filters, kernel_size, stride, padding

    def build(self, input_shape):
        c_in = input_shape[-1]
        self.kernel = self.add_weight(shape=(self.k, self.k, c_in, self.filters), initializer="he_normal")
        self.bias = self.add_weight(shape=(self.filters,), initializer="zeros")

    def call(self, X):
        windows = tf.image.extract_patches(X, sizes=[1, self.k, self.k, 1], strides=[1, self.stride, self.stride, 1],
                                           rates=[1, 1, 1, 1], padding=self.padding.upper())   # (N, H', W', k·k·C)
        return windows @ tf.reshape(self.kernel, (-1, self.filters)) + self.bias

class OurMaxPool2D(layers.Layer):
    def call(self, X):
        n_h, n_w, c = X.shape[1] // 2, X.shape[2] // 2, X.shape[3]
        blocks = tf.reshape(X[:, :2 * n_h, :2 * n_w, :], (-1, n_h, 2, n_w, 2, c))  # every 2×2 block side by side
        return tf.reduce_max(blocks, axis=(2, 4))

## Check: same weights ➜ same output as Keras's built-in layers

In [ ]:
x = x_train[:4]
for padding, stride in [("same", 1), ("valid", 1), ("same", 2)]:
    ours, builtin = OurConv2D(8, 3, stride, padding), layers.Conv2D(8, 3, strides=stride, padding=padding)
    out_ours = ours(x)
    builtin(x)                                                    # creates the built-in layer's weights
    builtin.set_weights(ours.get_weights())                       # same kernel and bias in both
    print(f"Conv2D padding={padding:<5} stride={stride}: output {tuple(out_ours.shape)} | equal to built-in? "
          f"{np.allclose(out_ours, builtin(x), atol=1e-5)}")

maps = OurConv2D(8)(x)
print(f"\nMaxPool2D: {tuple(maps.shape)} ➜ {tuple(OurMaxPool2D()(maps).shape)} | equal to built-in? "
      f"{np.allclose(OurMaxPool2D()(maps), layers.MaxPooling2D(2)(maps))}")

## Keras learns a filter — the same experiment as the hand-written loop
In the convolution-basics notebook, our own gradient loop found the Sobel filter from 500 digits. In Keras that is a 1-filter model, a loss and `fit`:

In [ ]:
digits = x_train[:500]
targets = tf.nn.conv2d(digits, sobel[:, :, None, None], strides=1, padding="VALID")   # what Sobel gives: (500, 26, 26, 1)

learner = keras.Sequential([keras.Input(shape=(28, 28, 1)), layers.Conv2D(1, 3, padding="valid")])
learner.compile(optimizer=keras.optimizers.SGD(0.5), loss="mse")   # mse = mean(delta²); lr 0.5 = lr 1.0 on ½·mean(delta²)
print("start (random):", np.round(learner.layers[0].get_weights()[0][:, :, 0, 0].astype(float), 1).tolist())
history = learner.fit(digits, targets, batch_size=500, epochs=300, verbose=0).history
kernel, bias = learner.layers[0].get_weights()
print(f"after 300 steps: {np.round(kernel[:, :, 0, 0].astype(float), 1).tolist()} | bias {bias[0]:.3f} | loss {history['loss'][0]:.3f} ➜ {history['loss'][-1]:.5f}")
print("Sobel:          ", sobel.tolist())

From a random start, 300 full-batch steps find **[[1.1, 0, −1], [1.9, 0.1, −2], [1.1, −0.1, −1]]** — the Sobel filter, just like our hand-written loop. Keras computed exactly the gradients we derived by hand (dL/dW = sum over all positions); we only described the model and the loss.

---
# Part 3: build and train a CNN from scratch (random start)
```
28×28×1 ─► Conv 3×3, 16 ─► ReLU ─► MaxPool ─► 14×14×16 ─► Conv 3×3, 32 ─► ReLU ─► MaxPool ─► 7×7×32 ─► Flatten (1568) ─► Dense 10
```
Now the target is not a filter output but the **label** (0–9): every filter is invented by the network.

In [ ]:
def our_cnn():
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        OurConv2D(16), layers.ReLU(), OurMaxPool2D(),
        OurConv2D(32), layers.ReLU(), OurMaxPool2D(),
        layers.Flatten(),
        layers.Dense(10),
    ])

LOSS = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
cnn = our_cnn()
cnn.summary()
cnn.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
start = time.time()
cnn.fit(x_train, y_train, validation_split=0.1, batch_size=64, epochs=3, verbose=2)
our_time = time.time() - start
print(f"\nour layers: test accuracy {100 * cnn.evaluate(x_test, y_test, verbose=0)[1]:.2f}% | {our_time:.0f} s for 3 epochs")

## The same network with Keras's built-in layers

In [ ]:
def builtin_cnn(filters=(16, 32), kernel=3, pool=True, input_shape=(28, 28, 1)):
    model = [keras.Input(shape=input_shape)]
    for f in filters:
        model += [layers.Conv2D(f, kernel, padding="same", activation="relu", kernel_initializer="he_normal")]
        model += [layers.MaxPooling2D(2)] if pool else []
    return keras.Sequential(model + [layers.Flatten(), layers.Dense(10)])

builtin = builtin_cnn()
builtin.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
start = time.time()
builtin.fit(x_train, y_train, validation_split=0.1, batch_size=64, epochs=3, verbose=0)
print(f"built-in layers: weights {builtin.count_params():,} (ours: {cnn.count_params():,}) | "
      f"test accuracy {100 * builtin.evaluate(x_test, y_test, verbose=0)[1]:.2f}% | {time.time() - start:.0f} s for 3 epochs")

Same network, same weight count, about the same accuracy. The built-in layer is faster (it uses an optimised convolution routine instead of copying every window).

---
# Part 4: CNN vs MLP — two tests
## Test 1: shifted digits
The shift test from the convolution-basics notebook, now with a trained CNN.

In [ ]:
def make_mlp():
    return keras.Sequential([keras.Input(shape=(28, 28, 1)), layers.Flatten(),
                             layers.Dense(128, activation="relu"), layers.Dense(10)])

mlp = make_mlp()
mlp.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
mlp.fit(x_train, y_train, batch_size=64, epochs=3, verbose=0)

print(f"weights: MLP {mlp.count_params():,} | CNN {builtin.count_params():,}\n")
print(f"{'shift (pixels right)':>20} | {'MLP':>6} | {'CNN':>6}")
print("-" * 40)
shift_results = {"MLP": [], "CNN": []}
for shift in range(5):
    shifted = np.roll(x_test, shift, axis=2)
    for name, model in [("MLP", mlp), ("CNN", builtin)]:
        shift_results[name].append(100 * model.evaluate(shifted, y_test, verbose=0)[1])
    print(f"{shift:>20} | {shift_results['MLP'][-1]:5.1f}% | {shift_results['CNN'][-1]:5.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, accs in shift_results.items():
    ax.plot(range(5), accs, "o-", label=name)
ax.set(xlabel="shift (pixels)", ylabel="test accuracy (%)", title="Shifted test digits (trained on unshifted digits)")
ax.legend()
plt.show()

## Test 2: scrambled pixels
In the convolution-basics notebook the MLP scored the **same** on digits whose pixels were shuffled (one fixed shuffle for all images). Now train **both** networks from scratch on scrambled digits. A CNN's 3×3 window now holds 9 random, unrelated pixels.

In [ ]:
order = np.random.default_rng(0).permutation(784)                  # one fixed shuffle for every image
def scramble(x):
    return x.reshape(len(x), 784)[:, order].reshape(-1, 28, 28, 1)

print(f"{'model':>5} | {'normal digits':>13} | {'scrambled digits':>16}")
print("-" * 42)
for name, make, normal_model in [("MLP", make_mlp, mlp), ("CNN", builtin_cnn, builtin)]:
    model = make()
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
    model.fit(scramble(x_train), y_train, batch_size=64, epochs=3, verbose=0)
    print(f"{name:>5} | {100 * normal_model.evaluate(x_test, y_test, verbose=0)[1]:12.2f}% | "
          f"{100 * model.evaluate(scramble(x_test), y_test, verbose=0)[1]:15.2f}%")

| | normal digits | scrambled digits | change |
|---|---|---|---|
| MLP | 96.80% | 97.09% | none (it never used the layout) |
| CNN | 98.24% | 95.05% | **−3.2 points** |

The CNN **loses** its advantage and falls below the MLP: its 3×3 windows now hold 9 unrelated pixels, so "neighbours" mean nothing and weight sharing reuses a filter on meaningless patches. It still learns something (95%) because the dense layer at the end can memorise pixel positions, as the MLP does. The CNN's strength **is** the 2D neighbourhood.

---
# Part 5: look inside — learned filters and feature maps

In [ ]:
first_filters = builtin.layers[0].get_weights()[0]                # (3, 3, 1, 16)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(first_filters[:, :, 0, k], cmap="RdBu_r")
    ax.set(title=f"filter {k}", xticks=[], yticks=[])
plt.suptitle("The 16 first-layer filters the CNN learned (red = positive, blue = negative)")
plt.tight_layout()
plt.show()

Do any of them look like the **hand-made** filters of the convolution-basics notebook? Compare each learned filter with each hand-made one (cosine similarity of the 9 numbers: 1 = same pattern, −1 = same pattern flipped in sign, 0 = unrelated).

In [ ]:
HAND = {
    "vertical edge":   sobel,
    "horizontal edge": sobel.T,
    "diagonal ↘":      np.array([[2, 1, 0], [1, 0, -1], [0, -1, -2]], dtype="float32"),
    "diagonal ↙":      np.array([[0, 1, 2], [-1, 0, 1], [-2, -1, 0]], dtype="float32"),
    "blur":            np.ones((3, 3), dtype="float32"),
    "centre spot":     np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype="float32"),
}
def cosine(a, b):
    return float(a.ravel() @ b.ravel() / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"{'learned filter':>14} | {'most similar hand-made filter':>30} | {'cosine':>6}")
print("-" * 58)
best_scores = []
for k in range(16):
    f = first_filters[:, :, 0, k]
    name, score = max(((n, cosine(f, h)) for n, h in HAND.items()), key=lambda t: abs(t[1]))
    best_scores.append(abs(score))
    print(f"{k:>14} | {name:>30} | {score:+.2f}")
random_scores = [max(abs(cosine(np.random.default_rng(i).normal(size=(3, 3)), h)) for h in HAND.values()) for i in range(1000)]
print(f"\nlearned filters: median best |cosine| {np.median(best_scores):.2f} | random 3×3 filters: {np.median(random_scores):.2f}")

Honest result: the learned filters are only **a little** more similar to the hand-made ones than random 3×3 filters are (median best |cosine| 0.64 vs 0.54 — with only 9 numbers, a random filter also matches something by chance). A few are clearly edge-like (filter 13: diagonal ↘ at 0.78, filter 11: diagonal ↙ at 0.74), most are **mixtures**. The network does not need clean textbook filters: it only needs a set of 16 that, together with the next layer, separates the digits. (VGG16's first layer in the convolution-basics notebook, trained on 1.4 million photos, had much cleaner edge maps.)

In [ ]:
inspect = keras.Model(builtin.inputs, [builtin.layers[0].output, builtin.layers[2].output])
maps1, maps2 = inspect([x_test[:1]])                           # inputs are passed as a list
print("layer 1 feature maps:", tuple(maps1.shape), "| layer 2 feature maps:", tuple(maps2.shape))

fig, axes = plt.subplots(3, 8, figsize=(14, 5.5))
axes[0, 0].imshow(x_test[0, :, :, 0], cmap="gray_r")
axes[0, 0].set_title("input")
for k in range(1, 8):
    axes[0, k].imshow(maps1[0, :, :, k], cmap="Reds")
    axes[0, k].set_title(f"layer 1, map {k}")
for k in range(8):
    axes[1, k].imshow(maps2[0, :, :, k], cmap="Reds")
    axes[1, k].set_title(f"layer 2, map {k}")
    axes[2, k].imshow(maps2[0, :, :, k + 8], cmap="Reds")
    axes[2, k].set_title(f"layer 2, map {k + 8}")
for ax in axes.ravel():
    ax.set(xticks=[], yticks=[])
plt.tight_layout()
plt.show()

Layer 1 maps (28×28) respond to strokes and edges; layer 2 maps (14×14) combine them into larger parts of the digit.

In [ ]:
pred = builtin.predict(x_test, verbose=0).argmax(axis=1)
wrong = np.where(pred != y_test)[0]
print(f"{len(wrong)} mistakes out of {len(y_test)}")
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, i in zip(axes, wrong[:10]):
    ax.imshow(x_test[i, :, :, 0], cmap="gray_r")
    ax.set(title=f"true {y_test[i]}, said {pred[i]}", xticks=[], yticks=[])
plt.show()

---
# Part 6: experiments (20,000 training images, 2 epochs each)

In [15]:
x_small, y_small = x_train[:20000], y_train[:20000]

def run(name, model):
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
    start = time.time()
    model.fit(x_small, y_small, batch_size=64, epochs=2, verbose=0)
    print(f"{name:>32} | {model.count_params():>8,} | {100 * model.evaluate(x_test, y_test, verbose=0)[1]:7.2f}% | {time.time() - start:5.1f} s")

print(f"{'model':>32} | {'weights':>8} | {'test acc':>8} | {'time':>6}")
print("-" * 66)
for filters in [(4, 8), (8, 16), (16, 32), (32, 64)]:
    run(f"filters {filters}", builtin_cnn(filters))
run("kernel 5×5 (16, 32)", builtin_cnn((16, 32), kernel=5))
run("1 conv block (16)", builtin_cnn((16,)))
run("3 conv blocks (16, 32, 64)", builtin_cnn((16, 32, 64)))
run("no pooling (16, 32)", builtin_cnn((16, 32), pool=False))

                 filters (8, 16) |    9,098 |   96.39% |   9.9 s
                filters (16, 32) |   20,490 |   97.61% |   7.1 s
                filters (32, 64) |   50,186 |   98.02% |  10.9 s
             kernel 5×5 (16, 32) |   28,938 |   97.86% |   9.6 s
               1 conv block (16) |   31,530 |   95.69% |   6.0 s
      3 conv blocks (16, 32, 64) |   29,066 |   97.31% |  10.1 s
             no pooling (16, 32) |  255,690 |   97.93% |   9.2 s


- **More filters** help up to a point: (4, 8) 94.7% ➜ (16, 32) 97.6% ➜ (32, 64) 98.0%, with 12× the weights for the last 0.4 points.
- **5×5 kernels** (97.9%) are only slightly better than 3×3 (97.6%), for 40% more weights.
- **One conv block** is clearly worse (95.7%): one layer of 3×3 filters sees only a 3×3 patch, too small for digit parts.
- **Three blocks** (97.2%) did not help here: after 3 poolings the maps are only 3×3, and 2 epochs is short for a deeper network.
- **No pooling** reaches 97.9% but with **255,690 weights** (12× more): the full-size 28×28×32 maps all go into the dense layer.

## Little data: where weight sharing pays off most
Train the MLP and the CNN on only 500, 2,000 or 10,000 digits (the same number of update steps each time, so small sets get more epochs).

In [ ]:
print(f"{'training images':>15} | {'epochs':>6} | {'MLP':>7} | {'CNN':>7}")
print("-" * 46)
small_data = {}
for n in [500, 2000, 10000]:
    epochs = 30000 // n
    row = []
    for make in [make_mlp, builtin_cnn]:
        keras.utils.set_random_seed(42)
        model = make()
        model.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
        model.fit(x_train[:n], y_train[:n], batch_size=64, epochs=epochs, verbose=0)
        row.append(100 * model.evaluate(x_test, y_test, verbose=0)[1])
    small_data[n] = row
    print(f"{n:>15,} | {epochs:>6} | {row[0]:6.2f}% | {row[1]:6.2f}%")

| training images | MLP | CNN | CNN ahead by |
|---|---|---|---|
| 500 | 85.0% | 91.1% | **6.1 points** |
| 2,000 | 90.1% | 95.5% | 5.4 points |
| 10,000 | 92.9% | 97.2% | 4.3 points |

The CNN wins at every size, and the gap is **largest when data is smallest**. With 500 images the MLP must learn 101,770 weights, one per pixel position; the CNN has 20,490 and every filter weight is trained at every position of every image — far more examples per weight.

---
# Part 7: use case (a) — sorting product photos for an online shop
A shop receives thousands of new product photos a day and each needs a category. **Fashion-MNIST** (Zalando product photos, 28×28 grayscale, 10 categories) is exactly this, and **much harder** than digits: a shirt, a pullover and a coat look alike at 28×28.

In [ ]:
(f_train, fy_train), (f_test, fy_test) = keras.datasets.fashion_mnist.load_data()
f_train = (f_train / 255.0).astype("float32")[..., None]
f_test = (f_test / 255.0).astype("float32")[..., None]
CLASSES = ["T-shirt/top", "trouser", "pullover", "dress", "coat", "sandal", "shirt", "sneaker", "bag", "ankle boot"]
print("train", f_train.shape, "test", f_test.shape)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.4))
for c in range(10):
    for row in range(2):
        axes[row, c].imshow(f_train[np.where(fy_train == c)[0][row], :, :, 0], cmap="gray_r")
        axes[row, c].set(xticks=[], yticks=[])
    axes[0, c].set_title(CLASSES[c], fontsize=10)
plt.show()

Same MLP as before, and a slightly bigger CNN (32 and 64 filters, plus a dense layer with dropout). 5 epochs each.

In [ ]:
def shop_cnn():
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", activation="relu"), layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"), layers.Dropout(0.3),
        layers.Dense(10),
    ])

shop = {}
for name, make in [("MLP", make_mlp), ("CNN", shop_cnn)]:
    keras.utils.set_random_seed(42)
    model = make()
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss=LOSS, metrics=["accuracy"])
    start = time.time()
    model.fit(f_train, fy_train, validation_split=0.1, batch_size=64, epochs=5, verbose=0)
    shop[name] = model
    print(f"{name}: weights {model.count_params():>7,} | test accuracy {100 * model.evaluate(f_test, fy_test, verbose=0)[1]:.2f}% | {time.time() - start:.0f} s")

In [ ]:
pred_mlp = shop["MLP"].predict(f_test, verbose=0).argmax(1)
logits = shop["CNN"].predict(f_test, verbose=0)
pred_cnn = logits.argmax(1)

print(f"{'category':>12} | {'MLP':>6} | {'CNN':>6}")
print("-" * 32)
for c in range(10):
    is_c = fy_test == c
    print(f"{CLASSES[c]:>12} | {100 * np.mean(pred_mlp[is_c] == c):5.1f}% | {100 * np.mean(pred_cnn[is_c] == c):5.1f}%")

confusion = np.zeros((10, 10), dtype=int)
np.add.at(confusion, (fy_test, pred_cnn), 1)
pairs = sorted(((confusion[a, b], a, b) for a in range(10) for b in range(10) if a != b), reverse=True)[:5]
print("\nCNN's most common mistakes:")
for count, a, b in pairs:
    print(f"   {CLASSES[a]:>12} taken for {CLASSES[b]:<12} {count:>4} times (out of 1,000)")

**What we saw:** CNN **91.2%** vs MLP 86.9% overall. The CNN gains most on the confusable clothes (pullover +10.5, coat +6.7, shirt +8.7 points), where shapes and textures matter. **Shirt** stays hard even for the CNN (63%): at 28×28 grayscale a shirt, a T-shirt, a coat and a pullover differ only by collar, buttons and sleeves. That is a limit of the **data**, not just of the model. (Dress is the one category the MLP got slightly better, 94.2% vs 93.0%.)

**In a real shop** the model does not have to decide everything. It tags the photos it is **sure** about and sends the rest to a person. The softmax probability tells us how sure it is:

In [ ]:
probs = tf.nn.softmax(logits).numpy()
confidence = probs.max(1)
print(f"{'auto-tag if sure above':>22} | {'photos auto-tagged':>18} | {'accuracy on those':>17} | {'sent to a person':>16}")
print("-" * 84)
for threshold in [0.0, 0.8, 0.9, 0.95, 0.99]:
    sure = confidence >= threshold
    print(f"{threshold:>22.2f} | {100 * sure.mean():17.1f}% | {100 * np.mean(pred_cnn[sure] == fy_test[sure]):16.2f}% | {100 * (1 - sure.mean()):15.1f}%")

wrong = np.where(pred_cnn != fy_test)[0][:8]
fig, axes = plt.subplots(1, 8, figsize=(15, 2.6))
for ax, i in zip(axes, wrong):
    ax.imshow(f_test[i, :, :, 0], cmap="gray_r")
    ax.set(xticks=[], yticks=[])
    ax.set_title(f"true {CLASSES[fy_test[i]]}\nsaid {CLASSES[pred_cnn[i]]} ({100 * confidence[i]:.0f}%)", fontsize=9)
plt.show()

| rule | auto-tagged | accuracy on auto-tagged | to a person |
|---|---|---|---|
| tag everything | 100% | 91.2% | 0% |
| tag if ≥ 90% sure | 81% | 97.8% | 19% |
| tag if ≥ 95% sure | 76% | **98.5%** | 24% |
| tag if ≥ 99% sure | 64% | 99.4% | 36% |

This is how such a model is really used: at "95% sure" three quarters of the photos are tagged automatically with 98.5% accuracy, and people check only the hard quarter. The shop chooses the threshold from what a wrong tag costs vs what a person's time costs. (The mistakes in the picture stay within a family — shoes taken for other shoes, tops and dresses for other tops — and some are made with high confidence: an ankle boot called a sandal at 100%. Confidence is useful, but not perfect.)

---
# Part 8: use case (b) — reading a handwritten number
A form has a box for a 6-digit PIN code, written by hand. The CNN from Part 3 reads **one** digit at a time, so the steps are:
1. **split** the strip into digits: columns with no ink are gaps between digits
2. **crop** each digit and put it in a 28×28 box (fit it into 20×20, then pad)
3. **classify** each digit with the CNN

To test it, we write numbers with **test** digits the CNN has never seen, with random gaps and a little up/down wobble.

In [ ]:
def write_number(indices, rng):
    # paste test digits side by side on one strip, with random gaps and vertical wobble
    pieces = [np.zeros((40, rng.integers(4, 10)))]
    for i in indices:
        img = x_test[i, :, :, 0]
        cols = np.where(img.max(axis=0) > 0)[0]
        img = img[:, cols[0]:cols[-1] + 1]                         # remove empty columns around the digit
        canvas = np.zeros((40, img.shape[1]))
        top = rng.integers(0, 13)
        canvas[top:top + 28] = img
        pieces += [canvas, np.zeros((40, rng.integers(3, 10)))]
    return np.concatenate(pieces, axis=1)

def split_digits(strip, min_gap=0):
    # runs of columns with ink = digits; runs closer than min_gap columns are joined into one digit
    ink = strip.max(axis=0) > 0
    edges = np.diff(np.concatenate([[0], ink.astype(int), [0]]))
    runs = [[a, b] for a, b in zip(np.where(edges == 1)[0], np.where(edges == -1)[0])]
    joined = [runs[0]]
    for a, b in runs[1:]:
        if a - joined[-1][1] < min_gap:
            joined[-1][1] = b
        else:
            joined.append([a, b])
    return [strip[:, a:b] for a, b in joined]

def to_mnist_box(piece, centre="box"):
    rows, cols = np.where(piece > 0)
    piece = piece[rows.min():rows.max() + 1, cols.min():cols.max() + 1]
    h, w = piece.shape
    scale = 20 / max(h, w)
    small = tf.image.resize(piece[..., None], (max(1, round(h * scale)), max(1, round(w * scale))), antialias=True).numpy()[:, :, 0]
    if centre == "box":                                            # middle of the box in the middle of 28×28
        top, left = (28 - small.shape[0]) // 2, (28 - small.shape[1]) // 2
    else:                                                          # centre of mass (where the ink is) in the middle, like MNIST
        yy, xx = np.indices(small.shape)
        top = int(round(14 - (yy * small).sum() / small.sum()))
        left = int(round(14 - (xx * small).sum() / small.sum()))
        top, left = min(max(top, 0), 28 - small.shape[0]), min(max(left, 0), 28 - small.shape[1])
    out = np.zeros((28, 28), dtype="float32")
    out[top:top + small.shape[0], left:left + small.shape[1]] = small
    return out

def read_number(strip, model, centre="box", min_gap=0):
    boxes = np.stack([to_mnist_box(p, centre) for p in split_digits(strip, min_gap)])[..., None]
    probs = tf.nn.softmax(model(boxes)).numpy()
    return "".join(str(d) for d in probs.argmax(1)), probs.max(1), boxes

def show_reading(strip, reading, sure, boxes, title):
    fig = plt.figure(figsize=(13, 4))
    ax = fig.add_subplot(2, 1, 1)
    ax.imshow(strip, cmap="gray_r")
    ax.set(title=title, xticks=[], yticks=[])
    for k in range(len(boxes)):
        ax = fig.add_subplot(2, len(boxes), len(boxes) + k + 1)
        ax.imshow(boxes[k, :, :, 0], cmap="gray_r")
        ax.set(title=f"read {reading[k]} ({100 * sure[k]:.0f}%)", xticks=[], yticks=[])
    plt.tight_layout()
    plt.show()

rng = np.random.default_rng(7)
example_idx = rng.choice(len(x_test), 6, replace=False)
example_strip = write_number(example_idx, rng)
reading, sure, boxes = read_number(example_strip, builtin)
print("written:", "".join(str(y_test[i]) for i in example_idx), "| CNN reads:", reading)
show_reading(example_strip, reading, sure, boxes, "the handwritten PIN code (one strip) — first version: digit box centred")

How often is the **whole** number right? Read 1,000 random 6-digit numbers:

In [ ]:
def test_reader(centre, min_gap, n=1000):
    rng = np.random.default_rng(0)
    digits_right = numbers_right = split_wrong = 0
    for _ in range(n):
        idx = rng.choice(len(x_test), 6, replace=False)
        truth = "".join(str(y_test[i]) for i in idx)
        reading, _, _ = read_number(write_number(idx, rng), builtin, centre, min_gap)
        if len(reading) != 6:
            split_wrong += 1
            continue
        digits_right += sum(a == b for a, b in zip(reading, truth))
        numbers_right += reading == truth
    return split_wrong, 100 * digits_right / (6 * (n - split_wrong)), 100 * numbers_right / n

first = test_reader("box", 0)
print(f"split into the wrong number of pieces: {first[0]} of 1,000")
print(f"single digits read correctly         : {first[1]:.2f}%")
print(f"whole 6-digit numbers fully correct  : {first[2]:.1f}%")
print(f"(on the original 28×28 test images the CNN gets {100 * builtin.evaluate(x_test, y_test, verbose=0)[1]:.2f}% per digit)")

**First version: worse than expected.** The CNN reads 98.24% of the original test digits, but only **92.2%** of the same digits cut out of a strip, and only **58%** of 6-digit numbers are fully right (one mistake in 6 digits spoils the number). 57 numbers were also split into the wrong number of pieces. Is the CNN the problem, or our preparation? Test the preparation alone: take the **original** test digits, put each through `to_mnist_box`, and classify again.

In [ ]:
reboxed = {centre: np.stack([to_mnist_box(img, centre) for img in x_test[:, :, :, 0]])[..., None] for centre in ["box", "mass"]}
print(f"original test images               : {100 * builtin.evaluate(x_test, y_test, verbose=0)[1]:.2f}%")
print(f"same images, re-boxed, box centred : {100 * builtin.evaluate(reboxed['box'], y_test, verbose=0)[1]:.2f}%")
print(f"same images, re-boxed, mass centred: {100 * builtin.evaluate(reboxed['mass'], y_test, verbose=0)[1]:.2f}%")

i = 0
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, img, title in [(axes[0], x_test[i, :, :, 0], "MNIST original"), (axes[1], reboxed["box"][i, :, :, 0], "box centred"),
                       (axes[2], reboxed["mass"][i, :, :, 0], "centre of mass")]:
    ax.imshow(img, cmap="gray_r")
    ax.axhline(13.5, color="red", lw=0.5)
    ax.axvline(13.5, color="red", lw=0.5)
    ax.set(title=title, xticks=[], yticks=[])
plt.show()

**Found it.** Re-boxing the very same digits drops accuracy from 98.24% to **91.8%**: the CNN was trained on MNIST, and MNIST puts the **centre of mass** of the ink in the middle, not the middle of the box. In the picture, the box-centred 7 sits higher than the original: for this CNN that is a different-looking input. Centring by mass gives back **98.24%** exactly.

The split errors have a second cause: some handwritten digits have a **gap inside them** (a 5 whose top bar does not touch, a broken 7), so the column rule cuts them into two pieces. Joining pieces closer than 3 columns fixes most of that. Compare all three versions:

In [ ]:
print(f"{'version':>40} | {'split wrong':>11} | {'digits right':>12} | {'numbers right':>13}")
print("-" * 88)
for name, centre, min_gap in [("1. box centred", "box", 0),
                              ("2. centre of mass (like MNIST)", "mass", 0),
                              ("3. + join pieces closer than 3 columns", "mass", 3)]:
    split_wrong, digits_pct, numbers_pct = first if (centre, min_gap) == ("box", 0) else test_reader(centre, min_gap)
    print(f"{name:>40} | {split_wrong:>4} / 1000 | {digits_pct:11.2f}% | {numbers_pct:12.1f}%")

reading, sure, boxes = read_number(example_strip, builtin, "mass", 3)
print("\nexample again — written:", "".join(str(y_test[i]) for i in example_idx), "| CNN reads:", reading)
show_reading(example_strip, reading, sure, boxes, "same strip, final version: centre of mass + join close pieces")

| version | split wrong | digits right | whole numbers right |
|---|---|---|---|
| 1. box centred | 57 / 1000 | 92.2% | 58.1% |
| 2. centre of mass (like MNIST) | 57 / 1000 | **98.3%** | 85.1% |
| 3. + join close pieces | **13** / 1000 | 98.3% | **88.8%** |

**Lesson: the input at use time must be prepared exactly like the training data.** The CNN never changed — only our cutting and centring — and whole-number accuracy went from 58% to 89%. The example 483189, misread as 483188 at first, is now read correctly.

Honest limits: our test strips are made of separate MNIST digits with gaps of 3+ columns. Real handwriting has **touching** digits that no column rule can split; reading those needs a model that reads the whole strip as a **sequence** (CNN features + an RNN, as in the RNN and LSTM folders).

---
# Summary

| Question | Answer from this notebook |
|---|---|
| Is Keras `Conv2D` the convolution we wrote by hand? | Yes — same output, same weight formula |
| Can Keras learn a filter like our hand-written loop? | Yes — 1 filter, `fit`, Sobel recovered |
| Why does a CNN beat an MLP on images? | It uses the 2D neighbourhood (scrambled pixels: CNN −3.2 points, MLP no change) and shares weights (5× fewer weights, +6 points with 500 images) |
| What does it learn? | Small filters — a few edge-like, most mixtures; deeper maps combine them into parts of the digit |
| Real use? | Product categories: 91% overall, 98.5% on the 76% it is sure about; handwritten PIN codes: 89% fully right once the digits are prepared like MNIST |

Next notebook: **CNNs on real colour photos (CIFAR-10) in Keras**, with data augmentation and Batch Normalisation.